# Week 36, Day 4: Layered architecture

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from dataclasses import dataclass

@dataclass
class Book:                                   # domain: rules, no input or output
    isbn: str
    title: str
    on_loan: bool = False

class MemoryBooks:                            # infrastructure: where books are kept
    def __init__(self):
        self._books = {}
    def add(self, book):
        self._books[book.isbn] = book
    def get(self, isbn):
        return self._books.get(isbn)

def borrow(books, isbn):                      # application: one use case
    book = books.get(isbn)
    if book is None or book.on_loan:
        return False
    book.on_loan = True
    return True

books = MemoryBooks()                         # the edge: wiring and presentation
books.add(Book("978-0", "Aké"))
for isbn in ["978-0", "978-0", "999"]:
    print("borrowed" if borrow(books, isbn) else "not available")

In [ ]:
import json
from dataclasses import dataclass, asdict
from typing import Protocol

@dataclass
class Book:
    isbn: str
    title: str
    on_loan_to: str | None = None

class BookRepository(Protocol):
    def add(self, book: Book) -> None: ...
    def get(self, isbn: str) -> Book | None: ...
    def save(self, book: Book) -> None: ...
    def all(self) -> list[Book]: ...

class MemoryBookRepository:
    def __init__(self):
        self._books = {}
    def add(self, book):
        self._books[book.isbn] = book
    def get(self, isbn):
        return self._books.get(isbn)
    def save(self, book):
        self._books[book.isbn] = book
    def all(self):
        return sorted(self._books.values(), key=lambda b: b.title)

class JsonBookRepository:
    def __init__(self, path):
        self.path = path
    def _load(self):
        try:
            with open(self.path) as f:
                return {d["isbn"]: Book(**d) for d in json.load(f)}
        except FileNotFoundError:
            return {}
    def _write(self, books):
        with open(self.path, "w") as f:
            json.dump([asdict(b) for b in books.values()], f, indent=1)
    def add(self, book):
        books = self._load()
        books[book.isbn] = book
        self._write(books)
    def get(self, isbn):
        return self._load().get(isbn)
    def save(self, book):
        self.add(book)
    def all(self):
        return sorted(self._load().values(), key=lambda b: b.title)

for repo in (MemoryBookRepository(), JsonBookRepository("books.json")):
    repo.add(Book("978-1", "Small Island"))
    repo.add(Book("978-0", "Aké"))
    b = repo.get("978-0")
    b.on_loan_to = "Ada"
    repo.save(b)
    print(type(repo).__name__, [(x.title, x.on_loan_to) for x in repo.all()])
print(JsonBookRepository("books.json").get("978-0"))

In [ ]:
from dataclasses import dataclass
from datetime import date, timedelta

class LibraryError(Exception):
    pass

class NotFound(LibraryError):
    pass

class OnLoan(LibraryError):
    pass

@dataclass
class Book:
    isbn: str
    title: str
    on_loan_to: str | None = None
    due: date | None = None

class MemoryBooks:
    def __init__(self, books=()):
        self._books = {b.isbn: b for b in books}
    def get(self, isbn):
        return self._books.get(isbn)
    def save(self, book):
        self._books[book.isbn] = book

class LoanService:                                   # application layer
    LOAN_DAYS = 14

    def __init__(self, books, today):
        self.books, self.today = books, today

    def borrow(self, isbn, member):
        book = self.books.get(isbn)
        if book is None:
            raise NotFound(isbn)
        if book.on_loan_to is not None:
            raise OnLoan(f"{book.title} is on loan until {book.due}")
        book.on_loan_to, book.due = member, self.today() + timedelta(days=self.LOAN_DAYS)
        self.books.save(book)
        return book.due

def handle(line, service):                            # presentation layer
    command, *args = line.split()
    try:
        if command == "borrow":
            due = service.borrow(args[0], args[1])
            return f"Borrowed. Please return it by {due:%d %B %Y}."
        return f"Unknown command: {command}"
    except NotFound as err:
        return f"No book with ISBN {err}."
    except LibraryError as err:
        return f"Sorry: {err}."

service = LoanService(MemoryBooks([Book("978-0", "Aké")]), today=lambda: date(2026, 11, 3))
for line in ["borrow 978-0 Ada", "borrow 978-0 Bo", "borrow 999 Bo", "renew 978-0"]:
    print(handle(line, service))

> **Key idea.** A layered architecture separates presentation (input and output for people), application services (use cases), the domain (core concepts and rules) and infrastructure (files, databases, clocks). The dependency rule: dependencies point inwards, so the domain depends on nothing and the details implement interfaces the inner layers define (Week 34's dependency inversion at program scale). A repository hides storage behind add, get, save and list methods, with in-memory and file versions. Services decide and raise domain errors; the presentation translates text to calls and errors to messages. Check layers by checking imports.

### Exercise: Which layer?

For each piece of code, give the layer it belongs in: `"presentation"`, `"application"`, `"domain"` or `"infrastructure"`.

In [ ]:
# p: a function that reads members.json and turns each entry into a Member
# q: a method Loan.is_overdue(today) that compares the due date with today
# r: a function that turns "borrow 978-0 Ada" into a call and prints the result
# s: a LoanService.renew(isbn) method that checks the rules and saves the loan
# t: a class that sends emails through a mail server
# u: the rule "a member may have at most five loans", as a function on Member

p = ___
q = ___
r = ___
s = ___
t = ___
u = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"infrastructure\", \"domain\", \"presentation\"), \"p to r\"\n    assert (s, t, u) == (\"application\", \"infrastructure\", \"domain\")")

### Exercise: Two repositories, one interface

Write two repositories for `Member` (given), each with `add(member)` (an id already present raises `ValueError`), `get(member_id)` (the member or `None`), `save(member)` (store changes; an unknown id raises `KeyError`) and `all()` (sorted by name). `MemoryMembers()` keeps them in a dictionary; `JsonMembers(path)` keeps them in a JSON file, so a second `JsonMembers` on the same path sees what the first stored. Both must give back members you can change without affecting what's stored until `save` is called.

In [ ]:
import json
from dataclasses import dataclass, asdict, replace

@dataclass
class Member:
    member_id: str
    name: str
    loans: int = 0

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def check(repo):\n    M = ns[\"Member\"]\n    repo.add(M(\"m2\", \"Tunde\"))\n    repo.add(M(\"m1\", \"Ada\"))\n    m = repo.get(\"m1\")\n    m.loans = 3\n    assert repo.get(\"m1\").loans == 0, \"changes shouldn't be stored until save\"\n    repo.save(m)\n    assert repo.get(\"m1\").loans == 3 and [x.name for x in repo.all()] == [\"Ada\", \"Tunde\"] and repo.get(\"zz\") is None\n    for f, err in [(lambda: repo.add(M(\"m1\", \"X\")), ValueError), (lambda: repo.save(M(\"nope\", \"Y\")), KeyError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass\ndef test_memory():\n    \"in memory\"\n    check(ns[\"MemoryMembers\"]())\ndef test_json():\n    \"in a file\"\n    check(ns[\"JsonMembers\"](\"members.json\"))\n    again = ns[\"JsonMembers\"](\"members.json\")\n    assert [(m.member_id, m.loans) for m in again.all()] == [(\"m1\", 3), (\"m2\", 0)]")

### Exercise: A service layer

Write a `LoanService(books, today)` where `books` is a repository (with `get` and `save`, as in the lesson) and `today` a function returning a `date`. It has three use cases, and must not print anything:

- `borrow(isbn, member)`: returns the due date (14 days on); raises `NotFound` for an unknown ISBN and `OnLoan` if it's already lent;
- `give_back(isbn)`: clears the loan and returns how many days late it was (0 if on time); raises `NotFound`, or `NotOnLoan` if it wasn't lent;
- `overdue()`: the titles of books past their due date, sorted.

`NotFound`, `OnLoan` and `NotOnLoan` are all kinds of `LibraryError`.

In [ ]:
from dataclasses import dataclass
from datetime import date, timedelta

@dataclass
class Book:
    isbn: str
    title: str
    on_loan_to: str | None = None
    due: date | None = None

class MemoryBooks:
    def __init__(self, books=()):
        self._books = {b.isbn: b for b in books}
    def get(self, isbn):
        return self._books.get(isbn)
    def save(self, book):
        self._books[book.isbn] = book
    def all(self):
        return list(self._books.values())

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_service():\n    \"use cases\"\n    from datetime import date as d\n    import io, contextlib\n    B, M, S = ns[\"Book\"], ns[\"MemoryBooks\"], ns[\"LoanService\"]\n    now = [d(2026, 11, 3)]\n    repo = M([B(\"1\", \"Ak\u00e9\"), B(\"2\", \"Kindred\"), B(\"3\", \"Small Island\")])\n    s = S(repo, lambda: now[0])\n    out = io.StringIO()\n    with contextlib.redirect_stdout(out):\n        assert s.borrow(\"1\", \"Ada\") == d(2026, 11, 17) and s.borrow(\"2\", \"Bo\") == d(2026, 11, 17)\n        now[0] = d(2026, 11, 20)\n        assert s.overdue() == [\"Ak\u00e9\", \"Kindred\"] and s.give_back(\"1\") == 3 and s.overdue() == [\"Kindred\"]\n        assert repo.get(\"1\").on_loan_to is None and repo.get(\"1\").due is None\n    assert out.getvalue() == \"\", \"the service shouldn't print\"\n    for f, err in [(lambda: s.borrow(\"9\", \"X\"), \"NotFound\"), (lambda: s.borrow(\"2\", \"X\"), \"OnLoan\"), (lambda: s.give_back(\"3\"), \"NotOnLoan\")]:\n        try:\n            f()\n            assert False, f\"should raise {err}\"\n        except ns[err]:\n            pass\n    assert all(issubclass(ns[e], ns[\"LibraryError\"]) for e in [\"NotFound\", \"OnLoan\", \"NotOnLoan\"])\n    s2 = S(M([B(\"4\", \"x\")]), lambda: d(2026, 1, 1))\n    s2.borrow(\"4\", \"A\")\n    assert s2.give_back(\"4\") == 0")

### Exercise: Checking the dependency rule

Write `violations(imports, layers)`. `imports` is a dictionary from a module name to the list of modules it imports; `layers` is a dictionary from module name to its layer number, where a **bigger number is further out** (for example domain 0, services 1, storage 2, cli 3). A module may import modules in its own layer or further in (a smaller or equal number); importing one further out is a violation. Modules not in `layers` (like the standard library) are ignored. Return the violations as `(module, imported)` pairs, sorted.

In [ ]:
def violations(imports, layers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_violations():\n    \"the rule\"\n    layers = {\"domain\": 0, \"services\": 1, \"storage\": 2, \"cli\": 3, \"main\": 4}\n    imports = {\"domain\": [\"dataclasses\", \"datetime\"], \"services\": [\"domain\", \"storage\"], \"storage\": [\"json\", \"domain\"],\n               \"cli\": [\"services\", \"domain\"], \"main\": [\"cli\", \"storage\", \"services\"], \"helpers\": [\"cli\"]}\n    assert violations(imports, layers) == [(\"services\", \"storage\")]\n    imports[\"domain\"].append(\"cli\")\n    assert violations(imports, layers) == [(\"domain\", \"cli\"), (\"services\", \"storage\")]\n    assert violations({}, layers) == []")

### Bug hunt: A domain that prints

The domain's `Book.lend` prints a message, so every use (including tests, and a future web page) gets text on the screen it didn't ask for: this prints `Aké lent to Ada` before `ok`. Domain code shouldn't do input or output. Remove the print from the domain, so the program prints only `ok` (the presentation decides what to show).

In [ ]:
class Book:
    def __init__(self, title):
        self.title = title
        self.on_loan_to = None

    def lend(self, member):
        if self.on_loan_to is not None:
            raise ValueError("already on loan")
        self.on_loan_to = member
        print(f"{self.title} lent to {member}")

b = Book("Aké")
b.lend("Ada")
print("ok")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a quiet domain\"\n    expect_output(\"ok\")\n    b = ns[\"Book\"](\"x\")\n    b.lend(\"A\")\n    assert b.on_loan_to == \"A\"")

### Bug hunt: Around the repository

The service looks books up in a global dictionary instead of the repository it was given, so a book added through the repository can't be found: this prints `not found`. Fix `find` to use the repository, so it prints `Kindred`.

In [ ]:
BOOKS = {"1": "Aké"}

class Repo:
    def __init__(self):
        self.books = {}
    def add(self, isbn, title):
        self.books[isbn] = title
    def get(self, isbn):
        return self.books.get(isbn)

class Service:
    def __init__(self, repo):
        self.repo = repo
    def find(self, isbn):
        return BOOKS.get(isbn, "not found")

repo = Repo()
repo.add("2", "Kindred")
print(Service(repo).find("2"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"through the repository\"\n    expect_output(\"Kindred\")\n    assert ns[\"Service\"](Repo()).find(\"1\") == \"not found\"")

### Bug hunt: A rule in the wrong layer

The presentation works out the due date itself, with its own (wrong) idea of the loan length, instead of using what the service returned: the screen says 10 November while the loan is really due on 17 November. Fix `show_borrow` so it reports the service's due date and prints `Due 17 November`.

In [ ]:
from datetime import date, timedelta

class Service:
    def borrow(self, today):
        return today + timedelta(days=14)

def show_borrow(service, today):
    service.borrow(today)
    due = today + timedelta(days=7)
    return f"Due {due.day} {due:%B}"

print(show_borrow(Service(), date(2026, 11, 3)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the service decides\"\n    expect_output(\"Due 17 November\")\n    assert \"timedelta\" not in source.split(\"def show_borrow\")[1].split(\"print(\")[0], \"the presentation shouldn't calculate the date\"")

### Puzzle: Presentation calling a service

Arrange the lines into a presentation function that calls the service and turns its errors into messages. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Repo:
    def __init__(self):
        self.d = {}
    def add(self, k, v):
        self.d[k] = v
    def get(self, k):
        return self.d.get(k)

class Service:
    def __init__(self, repo):
        self.repo = repo
    def double(self, k):
        v = self.repo.get(k)
        if v is None:
            raise KeyError(k)
        self.repo.add(k, v * 2)
        return v * 2

r = Repo()
r.add("a", 3)
s = Service(r)
s.double("a")

p = ___   # r.get("a")
q = ___   # s.double("a")
r_ = ___  # Service(Repo()).repo.get("a")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r_) == (6, 12, None)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a layered notes app

Build a small notes app in layers, all in one file but kept separate:

- **Domain**: `Note`, a dataclass with `note_id` (int), `title`, `body` and `tags` (a list of strings, stored lower case). `NotesError`, with `NoSuchNote` and `BadNote` as kinds of it.
- **Infrastructure**: `MemoryNotes` and `JsonNotes(path)` repositories with `add(note)`, `get(note_id)` (or `None`), `save(note)`, `delete(note_id)` and `all()` (sorted by id), handing out copies.
- **Application**: `NotesService(repo)` with `create(title, body, tags)` (a blank title raises `BadNote`; ids are 1 more than the largest so far, starting at 1; returns the id), `tag(note_id, tag)` (adds a tag in lower case if it isn't there), `find(tag)` (the titles of notes with that tag, sorted) and `remove(note_id)`. Unknown ids raise `NoSuchNote`.
- **Presentation**: `handle(line, service)` understanding `new <title> | <body> | <tag,tag>`, `tag <id> <tag>`, `find <tag>` and `rm <id>`, and returning text: `created note 3`, `tagged`, the titles joined by `, ` (or `nothing tagged x`), `removed`; a `NotesError` becomes `error: ` and its message; an unknown command gives `unknown command`.

The presentation must only call the service, and the service must only use its repository.

In [ ]:
import json
from dataclasses import dataclass, field, asdict, replace

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_layers():\n    \"the app\"\n    MN, S = ns[\"MemoryNotes\"], ns[\"NotesService\"]\n    s = S(MN())\n    assert handle(\"new Shopping | rice, oil | home,Food\", s) == \"created note 1\"\n    assert handle(\"new Rota | Ada Mon, Bo Tue | work\", s) == \"created note 2\"\n    assert handle(\"tag 2 HOME\", s) == \"tagged\" and handle(\"tag 2 home\", s) == \"tagged\"\n    assert handle(\"find home\", s) == \"Rota, Shopping\" and handle(\"find food\", s) == \"Shopping\" and handle(\"find gym\", s) == \"nothing tagged gym\"\n    assert handle(\"rm 1\", s) == \"removed\" and handle(\"rm 1\", s).startswith(\"error: \") and handle(\"dance\", s) == \"unknown command\"\n    assert handle(\"new  | x | y\", s).startswith(\"error: \") and handle(\"new Next | b | \", s) == \"created note 3\"\n    assert s.repo.get(2).tags == [\"work\", \"home\"]\ndef test_json():\n    \"file storage\"\n    JN, S = ns[\"JsonNotes\"], ns[\"NotesService\"]\n    s = S(JN(\"notes.json\"))\n    s.create(\"A\", \"x\", [\"T\"])\n    n = s.repo.get(1)\n    n.title = \"changed\"\n    assert s.repo.get(1).title == \"A\"\n    again = S(JN(\"notes.json\"))\n    assert again.create(\"B\", \"y\", []) == 2 and again.find(\"t\") == [\"A\"]\n    again.remove(1)\n    assert [n.note_id for n in JN(\"notes.json\").all()] == [2]\n    try:\n        again.tag(1, \"x\")\n        assert False\n    except ns[\"NoSuchNote\"]:\n        pass")

## Warm-up

Four small parts, each with one kind of job: what a book **is**, where books are **kept**, what the library **does**, and how the user **sees** it. Only the last part prints, and only the second knows how books are stored. That separation is what today is about.

## From classes to programs

Weeks 28 to 35 were about single classes and small groups of them. Real programs have dozens of modules, and the same design questions come back at a larger scale: which parts may know about which (coupling), and does each module have one kind of job (cohesion)?

Without an answer, programs drift into a shape where the code that draws the screen also reads the database and decides the business rules. Then changing the screen risks breaking the rules, testing the rules needs a database, and moving from files to a database means rewriting everything.

An **architecture** is a decision about the large-scale shape: what kinds of module there are, and what each may depend on. The most widely used is a **layered** architecture.

## Four layers and one rule

| Layer | Holds | In the library | Knows about |
|---|---|---|---|
| **presentation** | input and output for people: a command line, a web page, an app screen | reads commands, prints results and error messages | the application layer |
| **application** (services) | use cases: what the program does, step by step | "borrow a book", "return a book", "list overdue loans" | the domain, and interfaces for storage and other details |
| **domain** | the core concepts and rules | `Book`, `Loan`, "a book can't be lent twice", "loans last 14 days" | nothing outside itself |
| **infrastructure** | the details: files, databases, email, clocks | a JSON file of books, a real clock | the domain (to store its objects) |

And one rule above all, the **dependency rule**: **dependencies point inwards, towards the domain.** The domain imports nothing from the other layers; the application layer uses the domain and **interfaces** (Protocols, Week 32) for the details it needs; the infrastructure **implements** those interfaces; the presentation calls the application layer.

That's Week 34's dependency inversion applied to a whole program: the important rules (the domain and the use cases) don't depend on details (files, screens); details depend on them. Variations of this idea go by several names (Alistair Cockburn's "ports and adapters", Robert C. Martin's "clean architecture"), but they share this rule.

What it buys you:

- The domain and services can be **tested** with plain objects and fakes, in milliseconds.
- The presentation can **change** (a command line today, a web page next year) without touching the rules.
- Storage can **change** (a JSON file today, a database next year) without touching the rules.

## Repositories

A **repository** is an object that looks like a collection of domain objects (add one, get one, list them) while hiding where they're actually stored. The application layer depends on the repository's **interface**; the infrastructure provides implementations:

Notice `save`: the JSON version hands out **copies** loaded from the file, so changing a book changes nothing until it's saved. Code using a repository should always `save` what it changes, so it works with both kinds. The memory version is for tests and quick experiments; the file version is what the real program uses. Neither the books nor the code that uses the repository can tell which they have.

## Services and presentation

The **application layer** holds the use cases, usually as methods on a service class that's given its repository (and anything else it needs, such as a clock) when it's made. Services return results or raise **domain errors**; they never print or read input:

The division of work is sharp:

- the **service** decides (is it available? when is it due?) and reports problems as exceptions;
- the **presentation** translates: text into calls, results into friendly text, exceptions into messages;
- the **domain** errors (`NotFound`, `OnLoan`) are part of the domain's vocabulary, so the presentation can catch them without knowing how the service works.

## Layers in files

In a real project each layer is usually a module or a package, and you can check the dependency rule by looking at the imports:

| Module | May import |
|---|---|
| `library/domain.py` | the standard library only |
| `library/services.py` | `domain` |
| `library/storage.py` | `domain` |
| `library/cli.py` | `services`, `domain` (for its error types) |
| `main.py` (the composition root, Week 34) | everything, to wire it together |

If `domain.py` ever imports `cli` or `storage`, a layer has leaked. Some teams check this automatically in their tests or with tools that read imports.

Week 25's `spend` expense tracker already had this shape: pure functions for the totals and categories (domain), functions that loaded and saved files (infrastructure), and a command-line interface on top (presentation). Next week's project, a library system, will be built in exactly these layers.

Layers aren't free: a tiny script doesn't need four of them. Introduce them when a program grows beyond a few hundred lines, or when you can already see a second kind of storage or interface coming.

## Practice

## Quiz

## Daily challenge

## Before you go

Sketch the layers of a program you'd like to build: name two domain classes, two use cases, one kind of storage and one interface. Tomorrow: dependency injection at the scale of a whole program, building on Week 34.